# Junior IT Support / SysAdmin — Конспект

Практический конспект: **Windows → диагностика ПК → сети → Linux → troubleshooting → автоматизация**.

Главный принцип: **симптом → гипотеза → проверка → исправление → повторная проверка**.

## 1. Быстрые инструменты Windows

| Команда | Назначение |
|---|---|
| `Win + R → winver` | версия и сборка Windows |
| `Win + R → msinfo32` | подробная информация о системе |
| `Win + R → devmgmt.msc` | устройства и драйверы |
| `Win + R → services.msc` | службы Windows |
| `Win + R → eventvwr.msc` | системные журналы и ошибки |
| `Win + R → ncpa.cpl` | сетевые адаптеры |
| `Win + R → diskmgmt.msc` | диски и разделы |
| `Ctrl + Shift + Esc` | Диспетчер задач |

### Когда использовать
- ПК тормозит → Task Manager.
- Нет сети → `ncpa.cpl` → `ipconfig` → `ping` → `nslookup`.
- Ошибки системы → Event Viewer.
- Не работает устройство → Device Manager.
- Нужно быстро узнать Windows → `winver`.

## 2. PowerShell — базовая диагностика

### Система
```powershell
Get-ComputerInfo | Select-Object WindowsProductName, WindowsVersion, OsArchitecture, CsName
```

### CPU
```powershell
Get-CimInstance Win32_Processor | Select-Object Name, NumberOfCores, NumberOfLogicalProcessors
```

### RAM
```powershell
Get-CimInstance Win32_PhysicalMemory | Select-Object Manufacturer, Capacity, Speed, PartNumber
```

### Диски
```powershell
Get-PhysicalDisk | Select-Object FriendlyName, MediaType, HealthStatus, OperationalStatus, Size
Get-Volume | Select-Object DriveLetter, FileSystem, SizeRemaining, Size
```

### Процессы
```powershell
Get-Process | Sort-Object CPU -Descending | Select-Object -First 10 Name, CPU, Id
```

### Службы
```powershell
Get-Service
Get-Service | Where-Object {$_.Status -eq 'Running'}
```

### Полезные конструкции
- `|` — передаёт результат следующей команде.
- `Where-Object` — фильтрует.
- `Sort-Object` — сортирует.
- `Select-Object` — выбирает поля или количество объектов.
- `Get-Help <команда> -Examples` — примеры использования.

## 3. Сети — фундамент IT Support

### Основные понятия
- **IP-адрес** — адрес устройства в сети.
- **Subnet Mask / Prefix** — определяет границы сети.
- **Default Gateway** — обычно адрес роутера, через который идёт трафик в другие сети.
- **DNS** — преобразует имя (`google.com`) в IP-адрес.
- **DHCP** — автоматически выдаёт IP, маску, gateway и DNS.
- **MAC** — аппаратный адрес сетевого интерфейса.
- **TCP** — надёжный транспорт с установлением соединения.
- **UDP** — без установления соединения, меньше накладных расходов.

### Модель диагностики
```text
Сетевой адаптер
      ↓
IP-конфигурация
      ↓
Default Gateway
      ↓
Интернет по IP
      ↓
DNS
      ↓
Приложение / сайт
```

## 4. Диагностика «нет интернета» в Windows

### Шаг 1 — адаптер
`Win + R → ncpa.cpl`

Проверить, включён ли Wi-Fi/Ethernet и есть ли подключение.

### Шаг 2 — IP
```cmd
ipconfig /all
```

Смотреть: IPv4, Subnet Mask, Default Gateway, DNS Servers, DHCP Server.

### Шаг 3 — роутер
```cmd
ping <Default-Gateway>
```

Если gateway не отвечает, ищем проблему между ПК и локальной сетью.

### Шаг 4 — интернет по IP
```cmd
ping 8.8.8.8
```

### Шаг 5 — DNS
```cmd
nslookup google.com
ping google.com
```

Если `ping 8.8.8.8` работает, а имя не разрешается — подозреваем DNS.

### Дополнительно
```cmd
tracert 8.8.8.8
ipconfig /flushdns
ipconfig /release
ipconfig /renew
```

**Важно:** не применять `flushdns`, `release` или `renew` без понимания причины проблемы.

## 5. Linux — базовая навигация

| Команда | Назначение |
|---|---|
| `pwd` | текущий каталог |
| `ls -lah` | список файлов, включая скрытые |
| `cd` | перейти в каталог |
| `cp` | копировать |
| `mv` | переместить/переименовать |
| `rm` | удалить |
| `mkdir` | создать каталог |
| `cat` | вывести файл |
| `less` | читать файл постранично |
| `grep` | искать строки |
| `find` | искать файлы |
| `man` | документация |

Пример:
```bash
grep -i error /var/log/messages
```

Не используй `rm -rf` без понимания пути и последствий.

## 6. Linux — диагностика системы

### ОС и ядро
```bash
hostnamectl
cat /etc/os-release
uname -a
```

### CPU / RAM
```bash
lscpu
free -h
top
```

### Диски
```bash
lsblk -o NAME,SIZE,FSTYPE,TYPE,MOUNTPOINTS
df -h
```

Разница:
- `lsblk` — структура физических/логических блоковых устройств.
- `df -h` — занятое и свободное место смонтированных файловых систем.

## 7. Linux — сеть

Современная базовая команда:
```bash
ip addr
ip route
```

Проверка:
```bash
ping -c 4 8.8.8.8
ping -c 4 google.com
```

DNS:
```bash
resolvectl status
resolvectl query google.com
```

Порты и слушающие сервисы:
```bash
ss -tulpn
```

Маршрут:
```bash
tracepath 8.8.8.8
```

На некоторых системах отдельные утилиты могут отсутствовать; при необходимости их можно установить из репозиториев дистрибутива.

## 8. Linux — службы и логи

### systemd
```bash
systemctl status sshd
systemctl start sshd
systemctl stop sshd
systemctl restart sshd
systemctl enable sshd
```

### Логи
```bash
journalctl -b
journalctl -p err -b
journalctl -u sshd
```

Модель:
```text
Симптом → service status → journal → причина → исправление → повторная проверка
```

## 9. Linux — права и пользователи

```bash
id
whoami
ls -l
```

Изменение владельца:
```bash
sudo chown user:group file
```

Права:
```bash
chmod 644 file
chmod +x script.sh
```

Запомнить:
- `r` = read
- `w` = write
- `x` = execute

## 10. SSH — обязательный навык

Подключение:
```bash
ssh user@server-ip
```

Проверка доступности порта SSH:
```bash
ss -tlnp | grep ':22'
```

Главная идея:
`SSH → аутентификация → shell → диагностика → исправление`.

## 11. Troubleshooting — главный навык

Не угадываем причину. Работаем по гипотезам.

### Пример: «Linux-сервер не открывает сайт»
1. Есть ли IP? → `ip addr`
2. Есть ли маршрут? → `ip route`
3. Есть ли связь? → `ping`
4. Слушает ли веб-сервер порт? → `ss -tulpn`
5. Работает ли сервис? → `systemctl status nginx`
6. Что говорят логи? → `journalctl -u nginx`
7. Локально отвечает ли HTTP? → `curl -I http://localhost`
8. Проверить firewall/DNS/reverse proxy.
9. Повторить тест после исправления.

## 12. Автоматизация

Цель админа — постепенно превращать повторяющиеся действия в скрипты.

### PowerShell
```powershell
Get-Process | Sort-Object CPU -Descending | Select-Object -First 10 Name, CPU, Id
```

### Bash
```bash
df -h
free -h
uptime
```

Следующий уровень: собственные скрипты для отчёта о ПК/сервере, Bash, PowerShell и затем Ansible.

## 13. Минимальный набор Junior IT Support

### Windows
- `winver`, `msinfo32`
- Device Manager
- Services
- Event Viewer
- Task Manager
- PowerShell basics

### Сети
- IP / subnet / gateway / DNS / DHCP
- `ipconfig`
- `ping`
- `nslookup`
- `tracert`
- понимание TCP/UDP и портов

### Linux
- файловая система и права
- процессы
- systemd
- journalctl
- SSH
- `ip`, `ss`, `curl`
- диски и файловые системы

### Мышление
**Не чинить наугад. Сначала собрать факты.**

## 14. Практические задачи

### Задача 1 — Windows
Пользователь говорит: «Компьютер тормозит».

Найти 5 процессов с максимальной загрузкой CPU и 5 процессов с максимальным использованием RAM.

### Задача 2 — сеть
Пользователь говорит: «Wi-Fi подключён, но сайты не открываются».

Пройти цепочку:
`ncpa.cpl → ipconfig /all → gateway → 8.8.8.8 → nslookup google.com`.

### Задача 3 — Linux
Сервис не работает.

Проверить:
`systemctl status <service> → journalctl -u <service> → исправление → повторная проверка`.

### Задача 4 — диски
Найти файловую систему с критически малым свободным местом.

Linux: `df -h`.
Windows: `Get-Volume`.